# 03 — Giro de Estoque, Ponto de Reposicao e Lead Time
**Dataset:** Inventory Analysis Case Study  
**Objetivo:** Calcular o giro de estoque por produto, identificar o ponto ideal de reposicao e analisar o lead time dos fornecedores.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

beg   = pd.read_csv('../data/raw/BegInvFINAL12312016.csv')
end   = pd.read_csv('../data/raw/EndInvFINAL12312016.csv')
sales = pd.read_csv('../data/raw/SalesFINAL12312016.csv')
abc   = pd.read_csv('../data/processed/abc_classificacao.csv')

# Carregar compras em chunks (arquivo grande)
purchases = pd.read_csv('../data/raw/PurchasesFINAL12312016.csv',
                        usecols=['Brand', 'Description', 'VendorNumber', 'PODate', 'ReceivingDate', 'Quantity', 'PurchasePrice'],
                        parse_dates=['PODate', 'ReceivingDate'])

print(f'Compras: {len(purchases):,} registros carregados')

Compras: 2,372,474 registros carregados


## 1. Giro de Estoque por Produto
> **Formula:** Giro = Receita de Vendas / Estoque Medio  
> Giro alto = produto vendendo bem. Giro baixo = capital parado.

In [2]:
beg['ValorEstoque'] = beg['onHand'] * beg['Price']
end['ValorEstoque'] = end['onHand'] * end['Price']

est_inicial = beg.groupby('Brand')['ValorEstoque'].sum().reset_index()
est_inicial.columns = ['Brand', 'EstoqueInicial']
est_final = end.groupby('Brand')['ValorEstoque'].sum().reset_index()
est_final.columns = ['Brand', 'EstoqueFinal']

receita_prod = sales.groupby('Brand')['SalesDollars'].sum().reset_index()
receita_prod.columns = ['Brand', 'Receita']

giro = est_inicial.merge(est_final, on='Brand').merge(receita_prod, on='Brand').merge(
    abc[['Brand', 'Description', 'Classe']], on='Brand'
)

giro['EstoqueMedio'] = (giro['EstoqueInicial'] + giro['EstoqueFinal']) / 2
giro['Giro'] = np.where(giro['EstoqueMedio'] > 0, giro['Receita'] / giro['EstoqueMedio'], 0)
giro = giro[giro['EstoqueMedio'] > 0].sort_values('Giro', ascending=False)

print(f'Giro medio geral: {giro["Giro"].mean():.2f}x')
print(f'Mediana do giro:  {giro["Giro"].median():.2f}x')
print(f'Produtos com giro > 5x: {(giro["Giro"] > 5).sum():,}')
print(f'Produtos com giro < 1x: {(giro["Giro"] < 1).sum():,}')

Giro medio geral: 0.21x
Mediana do giro:  0.14x
Produtos com giro > 5x: 4
Produtos com giro < 1x: 5,568


## 2. Distribuicao do Giro por Classe ABC

In [3]:
giro_abc = giro.groupby('Classe')['Giro'].agg(['mean', 'median', 'count']).round(2).reset_index()
giro_abc.columns = ['Classe', 'Giro Medio', 'Giro Mediano', 'Produtos']
print(giro_abc.to_string(index=False))

fig = px.box(
    giro[giro['Giro'] < giro['Giro'].quantile(0.95)],
    x='Classe', y='Giro',
    color='Classe',
    color_discrete_map={'A': '#2ecc71', 'B': '#f39c12', 'C': '#e74c3c'},
    title='Distribuicao do Giro de Estoque por Classe ABC',
    labels={'Giro': 'Giro de Estoque (x)', 'Classe': 'Classe ABC'}
)
fig.update_layout(height=450, showlegend=False)
fig.show()

Classe  Giro Medio  Giro Mediano  Produtos
     A        0.27          0.23      1330
     B        0.22          0.15      1544
     C        0.17          0.08      2792


## 3. Produtos com Baixo Giro e Alto Estoque (Alertas)

In [4]:
alertas = giro[(giro['Giro'] < 1) & (giro['EstoqueMedio'] > 500)].sort_values('EstoqueMedio', ascending=False).head(20)

fig = px.scatter(
    giro[(giro['Giro'] < 10) & (giro['EstoqueMedio'] < giro['EstoqueMedio'].quantile(0.95))],
    x='Giro', y='EstoqueMedio',
    color='Classe',
    color_discrete_map={'A': '#2ecc71', 'B': '#f39c12', 'C': '#e74c3c'},
    hover_name='Description',
    size='Receita',
    title='Giro vs. Estoque Medio por Produto (tamanho = receita)',
    labels={'Giro': 'Giro de Estoque', 'EstoqueMedio': 'Estoque Medio (R$)'}
)
fig.add_vline(x=1, line_dash='dash', line_color='gray', annotation_text='Giro = 1x')
fig.update_layout(height=500)
fig.show()

print(f'Alertas: {len(alertas)} produtos com giro < 1x e estoque > R$500')
print(f'Capital parado nestes produtos: R$ {alertas["EstoqueMedio"].sum():,.2f}')

Alertas: 20 produtos com giro < 1x e estoque > R$500
Capital parado nestes produtos: R$ 5,835,775.62


## 4. Lead Time dos Fornecedores

In [5]:
purchases_validos = purchases.dropna(subset=['PODate', 'ReceivingDate']).copy()
purchases_validos['LeadTime'] = (purchases_validos['ReceivingDate'] - purchases_validos['PODate']).dt.days
purchases_validos = purchases_validos[(purchases_validos['LeadTime'] >= 0) & (purchases_validos['LeadTime'] <= 60)]

lead_fornecedor = purchases_validos.groupby('VendorNumber')['LeadTime'].agg(
    LeadTimeMedio='mean',
    LeadTimeMediano='median',
    Pedidos='count'
).round(1).reset_index().sort_values('LeadTimeMedio')

lead_fornecedor = lead_fornecedor[lead_fornecedor['Pedidos'] >= 50]

print(f'Lead time medio geral: {purchases_validos["LeadTime"].mean():.1f} dias')
print(f'Lead time mediano:     {purchases_validos["LeadTime"].median():.1f} dias')

fig = px.histogram(
    purchases_validos,
    x='LeadTime',
    nbins=30,
    title='Distribuicao do Lead Time de Fornecedores (dias entre PO e Recebimento)',
    labels={'LeadTime': 'Lead Time (dias)', 'count': 'Numero de Pedidos'},
    color_discrete_sequence=['#2980b9']
)
fig.add_vline(x=purchases_validos['LeadTime'].mean(), line_dash='dash',
              annotation_text=f"Media: {purchases_validos['LeadTime'].mean():.1f}d",
              line_color='red')
fig.update_layout(height=420)
fig.show()

Lead time medio geral: 7.6 dias
Lead time mediano:     8.0 dias


## 5. Ponto de Reposicao por Produto
> **Formula:** Ponto de Reposicao = Demanda Diaria Media x Lead Time Medio

In [6]:
# Demanda diaria media (vendas de janeiro = 31 dias)
demanda_diaria = sales.groupby('Brand')['SalesQuantity'].sum().reset_index()
demanda_diaria['DemandaDiaria'] = demanda_diaria['SalesQuantity'] / 31
demanda_diaria.columns = ['Brand', 'VendasJan', 'DemandaDiaria']

lead_medio_geral = purchases_validos['LeadTime'].mean()

ponto_reposicao = demanda_diaria.merge(abc[['Brand', 'Description', 'Classe']], on='Brand')
ponto_reposicao['PontoReposicao'] = (ponto_reposicao['DemandaDiaria'] * lead_medio_geral).round(0)
ponto_reposicao['EstoqueSeguranca'] = (ponto_reposicao['DemandaDiaria'] * lead_medio_geral * 0.3).round(0)

# Cruzar com estoque atual
est_atual = end.groupby('Brand')['onHand'].sum().reset_index()
ponto_reposicao = ponto_reposicao.merge(est_atual, on='Brand', how='left')
ponto_reposicao['Repor Agora?'] = ponto_reposicao['onHand'] <= ponto_reposicao['PontoReposicao']

urgentes = ponto_reposicao[(ponto_reposicao['Repor Agora?']) & (ponto_reposicao['Classe'] == 'A')].sort_values('DemandaDiaria', ascending=False)

print(f'Lead time medio usado: {lead_medio_geral:.1f} dias')
print(f'Produtos que precisam reposicao AGORA: {ponto_reposicao["Repor Agora?"].sum():,}')
print(f'Destes, Classe A (criticos): {len(urgentes):,}')

urgentes[['Description', 'Classe', 'DemandaDiaria', 'PontoReposicao', 'onHand']].head(15)

Lead time medio usado: 7.6 dias
Produtos que precisam reposicao AGORA: 198
Destes, Classe A (criticos): 28


,Description,Classe,DemandaDiaria,PontoReposicao,onHand
1452,Bacardi Superior Rum Trav,A,134.129032,1022.0,397.0
5629,Sebastiani Znfdl Sonoma Cnty,A,53.935484,411.0,6.0
1646,Bacardi Superior Rum,A,43.193548,329.0,8.0
1276,Ciroc Apple Vodka,A,39.548387,301.0,7.0
3062,Cava Mistinguett Brut,A,30.838710,235.0,94.0
3738,Liberty School Pnt Nr CC,A,21.000000,160.0,14.0
448,Jack Daniels Tennessee Fire,A,18.161290,138.0,25.0
5331,Landmark Overlook Chard,A,12.322581,94.0,20.0
3820,Lincourt Lindsays Pnt Nr,A,12.096774,92.0,12.0
5239,Clos Pegase Merlot Napa,A,12.032258,92.0,8.0


## 6. Exportar Grafico Principal para o README

In [7]:
# Grafico de destaque: Giro por Classe ABC
giro_resumo = giro.groupby('Classe').agg(
    GiroMedio=('Giro', 'mean'),
    Produtos=('Brand', 'count')
).reset_index()

fig_export = px.bar(
    giro_resumo,
    x='Classe', y='GiroMedio',
    text='GiroMedio',
    color='Classe',
    color_discrete_map={'A': '#2ecc71', 'B': '#f39c12', 'C': '#e74c3c'},
    title='Giro Medio de Estoque por Classe ABC',
    labels={'GiroMedio': 'Giro Medio (x)', 'Classe': 'Classe ABC'}
)
fig_export.update_traces(texttemplate='%{text:.2f}x', textposition='outside')
fig_export.update_layout(
    height=420, width=700,
    showlegend=False,
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(size=14)
)
fig_export.write_image('../outputs/giro_por_classe_abc.png', scale=2)
print('Exportado: outputs/giro_por_classe_abc.png')
fig_export.show()

Exportado: outputs/giro_por_classe_abc.png


## 7. Conclusoes Finais

### O que os dados mostram:
- **Produtos Classe A giram mais rapido** e devem ter reposicao automatica e prioritaria.
- **Produtos Classe C com giro baixo** representam capital imobilizado sem retorno, candidatos a promocao ou retirada.
- O **lead time medio dos fornecedores** define o tempo minimo para planejar o ponto de reposicao.
- Varios produtos **Classe A ja estao abaixo do ponto de reposicao**, indicando risco de ruptura iminente.

### O que isso significa para o pequeno empresario:
- Sem esses calculos, a reposicao e feita no feeling, e o feeling chega tarde.
- Uma planilha simples com essas tres colunas (giro, ponto de reposicao, estoque atual) ja transforma a gestao de qualquer negocio.
- O grande varejo faz isso com sistemas de milhoes. Da para fazer a mesma coisa com Python e dados historicos.
